# Breast cancer: how should we choose the 16 candidate genes?

Side experiment. My clean pipeline is in `breast.ipynb`.

When I let the Wilcoxon test look at **all genes** instead of the 2,000 "highly variable" ones
(plus markers I added by hand), only 3 of the 16 candidates stayed the same.
So the way I build the candidate pool matters a lot. Here I compare three pools:

- **A. Markers + HVG**: 2,000 highly variable genes + hand-picked markers, then Wilcoxon (my original way)
- **B. Wilcoxon, all genes**: no hand-picking, the test looks at every gene
- **C. Lasso (ML)**: L1 logistic regression picks the genes

Each pool goes through the same QUBO and the same grading. I also add a pure ML competitor:
the lasso choosing 2 genes by itself.
Everything is chosen on the training half only.

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import time, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

from qiskit import transpile
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.optimize import minimize

## 1. Load the data (same steps as `breast.ipynb`)

In [14]:
info = pd.read_csv("data/GSE75688_final_sample_information.txt.gz", sep="\t")
tpm = pd.read_csv("data/GSE75688_GEO_processed_Breast_Cancer_raw_TPM_matrix.txt.gz", sep="\t")

info = info.set_index(info.columns[0])
sc_info = info[info["type"].astype(str).str.upper() == "SC"].copy()
lab = sc_info["index"].astype(str).str.lower()
sc_info["malignant"] = np.where(lab.str.startswith("non"), "1",
                        np.where(lab.str.contains("tumor"), "2", "0"))
sc_info["cell_type"] = sc_info["index3"].astype(str)

genes = tpm.set_index("gene_name")
cells = [c for c in sc_info.index if c in genes.columns]
adata = ad.AnnData(X=genes[cells].T.values.astype("float32"))
adata.obs_names = cells
adata.var_names = genes.index.astype(str)
adata.var_names_make_unique()
adata.obs["malignant"] = sc_info.loc[cells, "malignant"].values
adata.obs["cell_type"] = sc_info.loc[cells, "cell_type"].values
sc.pp.log1p(adata)

adata = adata[adata.obs["malignant"] != "0"].copy()
df = adata.to_df()                                   # cells x genes table (log scale)
y_all = (adata.obs["malignant"] == "2").astype(int).values
print(df.shape)

(515, 57915)


## 2. The same train / test split as `breast.ipynb`

Same seed (42), same stratification, so the training and test cells are identical to the clean notebook.
I also keep only genes seen (above 0) in at least 10 training cells. Rarer genes are too noisy to trust.

In [15]:
train_idx, test_idx = train_test_split(adata.obs_names, test_size=0.5,
                                       random_state=42, stratify=adata.obs["malignant"])
y = (adata[train_idx].obs["malignant"] == "2").astype(int).values
y_test = (adata[test_idx].obs["malignant"] == "2").astype(int).values

seen = (df.loc[train_idx] > 0).sum(axis=0)
kept = seen[seen >= 10].index
print(len(train_idx), "train,", len(test_idx), "test,", len(kept), "genes kept")

257 train, 258 test, 17348 genes kept


## 3. Candidate pools

A helper that runs the Wilcoxon test on the training cells, for any list of genes,
and returns the top 8 cancer genes + top 8 normal genes.

In [16]:
def wilcoxon_top(gene_list, k=8):
    a = adata[train_idx, list(gene_list)].copy()
    sc.tl.rank_genes_groups(a, groupby="malignant", method="wilcoxon")
    top_c = sc.get.rank_genes_groups_df(a, group="2").head(k)["names"]
    top_n = sc.get.rank_genes_groups_df(a, group="1").head(k)["names"]
    return list(top_c) + list(top_n)

**Pool A: markers + highly variable genes** (my original way)

In [17]:
hv = adata.copy()
sc.pp.highly_variable_genes(hv, n_top_genes=2000)
markers = ["PTPRC", "CD3E", "CD19", "CD14", "PECAM1", "COL1A1",
           "EPCAM", "KRT8", "KRT18", "KRT19", "ESR1", "ERBB2"]
markers = [g for g in markers if g in hv.var_names]
hv.var.loc[markers, "highly_variable"] = True
hv_genes = [g for g in hv.var_names[hv.var["highly_variable"]] if g in kept]

pool_A = wilcoxon_top(hv_genes)
pool_A

['EPCAM',
 'KRT19',
 'KRT18',
 'KRT8',
 'ERBB2',
 'CLU',
 'VEGFA',
 'TPM1',
 'RGS1',
 'ALOX5AP',
 'PTPRC',
 'GPR183',
 'CD48',
 'IGKC',
 'CARD16',
 'IFI30']

**Pool B: Wilcoxon on all genes** (no hand-picking)

In [18]:
pool_B = wilcoxon_top(kept)
pool_B

['PTPRF',
 'TFAP2C',
 'EPCAM',
 'ERBB3',
 'ESRP1',
 'ENAH',
 'MYO6',
 'KRT19',
 'TMSB4X',
 'RGS1',
 'CD53',
 'CD74',
 'HLA-DRB1',
 'IFI16',
 'LAPTM5',
 'HLA-DRB5']

**Pool C: lasso (ML)**

L1 logistic regression on all kept genes (training half only). The L1 penalty pushes most
gene weights to exactly zero, so it selects genes by itself. I make the penalty weaker step
by step until at least 8 genes have a positive weight (cancer side) and 8 a negative weight
(normal side), then take the 8 strongest on each side.
Genes are standardized first, so weights are comparable.

In [19]:
scaler = StandardScaler().fit(df.loc[train_idx, kept].values)
Xs = scaler.transform(df.loc[train_idx, kept].values)

for C in np.logspace(-3, 1, 40):
    lasso = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=5000).fit(Xs, y)
    coef = lasso.coef_[0]
    if (coef > 0).sum() >= 8 and (coef < 0).sum() >= 8:
        break
print("C =", round(C, 4), "| nonzero weights:", (coef != 0).sum())

kept_arr = np.array(kept)
pool_C = list(kept_arr[np.argsort(-coef)[:8]]) + list(kept_arr[np.argsort(coef)[:8]])
pool_C

C = 0.1125 | nonzero weights: 39


['PTPRF',
 'ESRP1',
 'TFAP2C',
 'COX6C',
 'ERBB3',
 'TMC4',
 'RP11-206M11.7',
 'MLPH',
 'HLA-DRB5',
 'HLA-DRB1',
 'CD53',
 'HLA-E',
 'ZEB2',
 'CXCR4',
 'TGM2',
 'HLA-DMA']

In [20]:
pools = {"A. markers + HVG": pool_A, "B. Wilcoxon all genes": pool_B, "C. lasso": pool_C}
print("A and B share:", sorted(set(pool_A) & set(pool_B)))
print("A and C share:", sorted(set(pool_A) & set(pool_C)))
print("B and C share:", sorted(set(pool_B) & set(pool_C)))

A and B share: ['EPCAM', 'KRT19', 'RGS1']
A and C share: []
B and C share: ['CD53', 'ERBB3', 'ESRP1', 'HLA-DRB1', 'HLA-DRB5', 'PTPRF', 'TFAP2C']


## 4. One function for the whole QUBO pipeline

Given 16 candidates: relevance r, positive-only redundancy, the Q matrix (k = 2, α = 1, λ = 2),
brute force over all 2^16 panels, and the baselines.
Same formulas as my other notebooks.

In [22]:
n, k, alpha, lam = 16, 2, 1.0, 2.0
const = lam * k**2
bits = (np.arange(2**n)[:, None] >> np.arange(n)) & 1

def build_Q(cands):
    X = df.loc[train_idx, cands].values
    corr = np.corrcoef(X.T)
    r = np.array([abs(np.corrcoef(X[:, i], y)[0, 1]) for i in range(n)])
    R = np.clip(corr, 0, None); np.fill_diagonal(R, 0)
    Q = alpha * R / 2 + lam * (np.ones((n, n)) - np.eye(n))
    Q[np.diag_indices(n)] += -r + lam * (1 - 2 * k)
    return Q, r

def brute_force(Q):
    scores = np.einsum("bi,ij,bj->b", bits, Q, bits) + const
    b = int(scores.argmin())
    return b, scores[b]

def evaluate(panel, tr=None, te=None, ytr=None, yte=None):
    tr = train_idx if tr is None else tr
    te = test_idx if te is None else te
    ytr = y if ytr is None else ytr
    yte = y_test if yte is None else yte
    model = LogisticRegression(max_iter=1000).fit(df.loc[tr, panel].values, ytr)
    return balanced_accuracy_score(yte, model.predict(df.loc[te, panel].values))

def ten_splits(panel):
    scores = []
    for seed in range(10):
        tr, te = train_test_split(df.index, test_size=0.5, random_state=seed, stratify=y_all)
        ytr = y_all[df.index.get_indexer(tr)]
        yte = y_all[df.index.get_indexer(te)]
        scores.append(evaluate(panel, tr, te, ytr, yte))
    return np.mean(scores), np.std(scores)

## 5. Run all three pools

In [24]:
results, Qs = [], {}
for name, cands in pools.items():
    Q, r = build_Q(cands)
    b, s = brute_force(Q)
    Qs[name] = (Q, cands, b, s)
    order = np.argsort(-r)
    panels = {
        "QUBO": [cands[i] for i in range(n) if bits[b, i]],
        "top 2 by relevance": [cands[order[0]], cands[order[1]]],
        "best single gene": [cands[order[0]]],
    }
    for method, p in panels.items():
        m, sd = ten_splits(p)
        results.append({"pool": name, "method": method, "panel": ", ".join(p),
                        "seed 42": round(evaluate(p), 3), "10 splits": f"{m:.3f} ± {sd:.3f}"})

results_df = pd.DataFrame(results)
results_df

,pool,method,panel,seed 42,10 splits
0,A. markers + HVG,QUBO,"KRT18, RGS1",0.916,0.902 ± 0.012
1,A. markers + HVG,top 2 by relevance,"KRT18, EPCAM",0.912,0.905 ± 0.012
2,A. markers + HVG,best single gene,KRT18,0.880,0.897 ± 0.011
3,B. Wilcoxon all genes,QUBO,"TFAP2C, CD53",0.889,0.895 ± 0.013
4,B. Wilcoxon all genes,top 2 by relevance,"TFAP2C, PTPRF",0.920,0.924 ± 0.010
5,B. Wilcoxon all genes,best single gene,TFAP2C,0.902,0.908 ± 0.009
6,C. lasso,QUBO,"TFAP2C, CD53",0.889,0.895 ± 0.013
7,C. lasso,top 2 by relevance,"TFAP2C, PTPRF",0.920,0.924 ± 0.010
8,C. lasso,best single gene,TFAP2C,0.902,0.908 ± 0.009


## 6. Pure ML competitor: the lasso picks 2 genes by itself

No QUBO at all. I make the L1 penalty strong enough that only about 2 genes survive,
searching over all kept genes, then take the 2 with the largest weights.
This is a fair classical rival for the QUBO.

In [11]:
for C in np.logspace(-4, 0, 80):
    l2 = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=5000).fit(Xs, y)
    if (l2.coef_[0] != 0).sum() >= 2:
        break
w = l2.coef_[0]
panel_lasso = list(kept_arr[np.argsort(-np.abs(w))[:2]])
m, sd = ten_splits(panel_lasso)
print("lasso 2 genes:", panel_lasso)
print("seed 42:", round(evaluate(panel_lasso), 3), "| 10 splits:", f"{m:.3f} ± {sd:.3f}")

lasso 2 genes: ['TFAP2C', 'PTPRF']
seed 42: 0.92 | 10 splits: 0.924 ± 0.010


## 7. QAOA on pool B (the automatic candidates)

Same as `breast.ipynb`: Hamiltonian check, p = 1 grid, p = 2 COBYLA.
Then I can compare with the QAOA numbers for pool A.

In [25]:
Q, cands, b_best, s_best = Qs["B. Wilcoxon all genes"]
terms, offset = [], const
for i in range(n):
    terms.append(("Z", [i], -Q[i, i] / 2))
    offset += Q[i, i] / 2
    for j in range(n):
        if i != j:
            terms.append(("ZZ", [i, j], Q[i, j] / 4))
            terms.append(("Z", [i], -Q[i, j] / 4))
            terms.append(("Z", [j], -Q[i, j] / 4))
            offset += Q[i, j] / 4
H = SparsePauliOp.from_sparse_list(terms, num_qubits=n).simplify()
diag = H.to_matrix(sparse=True).diagonal().real
best = int(np.argmin(diag))
print(len(H), "terms | lowest energy:", round(diag[best] + offset, 3),
      "| matches brute force:", best == b_best)

136 terms | lowest energy: -1.502 | matches brute force: True


In [28]:
sizes = bits.sum(axis=1)
def run(qc, angles):
    p = Statevector(qc.assign_parameters(angles)).probabilities()
    return p @ diag + offset, p

qc1 = transpile(QAOAAnsatz(H, reps=1), basis_gates=["h", "rz", "rx", "cx"])
betas, gammas = np.linspace(0, np.pi / 2, 25), np.linspace(0, 0.5, 25)
E = np.array([[run(qc1, [bt, g])[0] for bt in betas] for g in gammas])
a, bb = np.unravel_index(E.argmin(), E.shape)
angles1 = [betas[bb], gammas[a]]
e1, probs1 = run(qc1, angles1)

qc2 = transpile(QAOAAnsatz(H, reps=2), basis_gates=["h", "rz", "rx", "cx"])
res2 = minimize(lambda x: run(qc2, x)[0], [angles1[0], angles1[0], angles1[1], angles1[1]],
                method="COBYLA", options={"maxiter": 200})
e2, probs2 = run(qc2, res2.x)

pd.DataFrame({
    "avg energy": [offset, e1, e2, diag[best] + offset],
    "P(best panel)": [1 / 2**n, probs1[best], probs2[best], 1],
    "P(2 genes)": [(sizes == 2).mean(), probs1[sizes == 2].sum(), probs2[sizes == 2].sum(), 1],
}, index=["random", "QAOA p = 1", "QAOA p = 2", "exact"]).round(5)

,avg energy,P(best panel),P(2 genes)
random,83.71532,0.00002,0.00183
QAOA p = 1,58.28334,0.00001,0.00162
QAOA p = 2,0.59052,0.00405,0.46282
exact,-1.50193,1.00000,1.00000


## 8. What I learned

(Write here after running: which pool gave the best panel? Did the QUBO beat top-2 and the lasso?
Did hand-picked markers help or not?)

In [27]:
for a_ in [0, 0.02, 0.05, 0.1, 0.15, 0.2]:
    alpha = a_
    Q, r = build_Q(pool_B)
    b, s = brute_force(Q)
    panel = [pool_B[i] for i in range(n) if bits[b, i]]
    m, sd = ten_splits(panel)
    print(f"alpha = {a_:<5} {str(panel):28s} {m:.3f} ± {sd:.3f}")
alpha = 1.0          # put it back

alpha = 0     ['PTPRF', 'TFAP2C']          0.924 ± 0.010
alpha = 0.02  ['PTPRF', 'TFAP2C']          0.924 ± 0.010
alpha = 0.05  ['PTPRF', 'TFAP2C']          0.924 ± 0.010
alpha = 0.1   ['PTPRF', 'TFAP2C']          0.924 ± 0.010
alpha = 0.15  ['TFAP2C', 'CD53']           0.895 ± 0.013
alpha = 0.2   ['TFAP2C', 'CD53']           0.895 ± 0.013
